# Materialize Dimension Tables
NYC TLC Yellow Taxi

## Purpose
This notebook creates the **dimension tables** used for analytics and semantic modeling on top of the curated Yellow Taxi fact data.

It produces small, stable Delta tables that:
- Provide human-readable labels for coded fields (VendorID, RatecodeID, payment_type)
- Map Taxi Zone `LocationID` values to Borough/Zone/Service Zone
- Provide a robust Date dimension with common calendar attributes and US federal holidays (observed)

---

## What this notebook creates

### Dimension tables (Delta)
- `dim_vendor`
- `dim_ratecode`
- `dim_payment_type`
- `dim_taxi_zone`
- `dim_date`

---

## Inputs

### Curated fact table
- `tlc_yellow_tripdata_unified`

### Taxi Zone lookup file
A CSV file named `taxi_zone_lookup.csv` (official TLC lookup) uploaded to the lakehouse:
- `Files/mapping/taxi_zone_lookup.csv`

---

## Output usage
These dimensions are intended for:
- Power BI / Fabric semantic models (star schema)
- Consistent labels and grouping in reports
- Handling unknowns explicitly (`-1` members)

---

## Design notes

### Unknown members
Each coded dimension includes a standardized “Unknown / Not provided” member with key `-1`.
This ensures:
- No blank labels in the semantic model
- Stable relationships even when source values are null, out of range, or legacy artifacts

### Taxi zones
Taxi zone keys are conformed through `LocationID`.  
The fact table contains `PULocationID` and `DOLocationID`, which typically require two role-playing relationships in the semantic model (Pickup Zone and Dropoff Zone).

### Holidays
The Date dimension includes US federal holidays, including observed dates.  
Juneteenth is included from 2021 onwards.

---


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Row

## Configuration

Define input tables, lookup files, and the date range used for the Date dimension.

In [ ]:
# ----------------------------
# Configuration
# ----------------------------
FACT_TABLE = "tlc_yellow_tripdata_unified"
TAXI_ZONE_LOOKUP_PATH = "Files/mapping/taxi_zone_lookup.csv"

# Date range for dim_date (defaults based on your loaded history)
START_DATE = "2009-01-01"
END_DATE   = "2026-12-31"

## Vendor dimension

Creates a small, dictionary-aligned Vendor dimension.
An explicit **Unknown / Legacy Vendor (-1)** member is included to ensure stable joins.

In [ ]:
# ----------------------------
# 1) dim_vendor
# Dictionary-aligned + explicit Unknown member (-1)
# ----------------------------
vendor_rows = [
    Row(VendorID=1, vendor_name="Creative Mobile Technologies, LLC"),
    Row(VendorID=2, vendor_name="VeriFone / Curb Mobility, LLC"),
    Row(VendorID=6, vendor_name="Myle Technologies Inc"),
    Row(VendorID=7, vendor_name="Helix"),
    Row(VendorID=-1, vendor_name="Unknown"),
]
df_vendor = spark.createDataFrame(vendor_rows)

(
    df_vendor.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dim_vendor")
)

#display(df_vendor.orderBy("VendorID"))

## Rate code dimension

Maps `RatecodeID` values to human-readable rate descriptions.
Includes an **Unknown (-1)** member for invalid or missing codes.

In [ ]:
# ----------------------------
# 2) dim_ratecode
# ----------------------------
ratecode_rows = [
    Row(RatecodeID=1, ratecode_description="Standard rate"),
    Row(RatecodeID=2, ratecode_description="JFK"),
    Row(RatecodeID=3, ratecode_description="Newark"),
    Row(RatecodeID=4, ratecode_description="Nassau or Westchester"),
    Row(RatecodeID=5, ratecode_description="Negotiated fare"),
    Row(RatecodeID=6, ratecode_description="Group ride"),
    Row(RatecodeID=-1, ratecode_description="Unknown"),
]
df_ratecode = spark.createDataFrame(ratecode_rows)

(
    df_ratecode.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dim_ratecode")
)

#display(df_ratecode.orderBy("RatecodeID"))

## Payment type dimension

Maps payment type codes to readable labels.
An explicit **Unknown (-1)** member is added for robustness in the semantic model.

In [ ]:
# ----------------------------
# 3) dim_payment_type
# ----------------------------
payment_rows = [
    Row(payment_type=1, payment_description="Credit card"),
    Row(payment_type=2, payment_description="Cash"),
    Row(payment_type=3, payment_description="No charge"),
    Row(payment_type=4, payment_description="Dispute"),
    Row(payment_type=5, payment_description="Unknown"),
    Row(payment_type=6, payment_description="Voided trip"),
    Row(payment_type=-1, payment_description="Unknown"),
]
df_payment = spark.createDataFrame(payment_rows)

(
    df_payment.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dim_payment_type")
)

#display(df_payment.orderBy("payment_type"))

## Taxi zone dimension

Loads the official NYC TLC Taxi Zone lookup file and creates a conformed zone dimension.

The **Unknown (-1)** row ensures all pickup and dropoff LocationIDs can safely join.

In [ ]:
# ----------------------------
# 4) dim_taxi_zone
# Reads the official TLC taxi zone lookup CSV
# Adds Unknown member (-1)
# ----------------------------
df_zone = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(TAXI_ZONE_LOOKUP_PATH)
)

# Standardize columns and types
df_zone = (
    df_zone
    .withColumn("LocationID", F.col("LocationID").cast("int"))
    .withColumn("Borough", F.col("Borough").cast("string"))
    .withColumn("Zone", F.col("Zone").cast("string"))
    .withColumn("service_zone", F.col("service_zone").cast("string"))
    .dropDuplicates(["LocationID"])
)

# Add Unknown row (-1)
unknown_zone = spark.createDataFrame(
    [Row(LocationID=-1, Borough="Unknown", Zone="Unknown", service_zone="Unknown")]
)

df_zone = df_zone.unionByName(unknown_zone)

(
    df_zone.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dim_taxi_zone")
)

#display(df_zone.orderBy("LocationID"))

## Date dimension
Creates a full calendar Date dimension with:
- Year, quarter, month, week, weekday attributes
- ISO weekday numbering (Monday = 1)
- Month start / end flags
- US federal holidays (observed)

In [ ]:
# ----------------------------
# 5) dim_date (calendar + US federal holidays, observed)
# ----------------------------

# Base date dimension
df_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{START_DATE}'), to_date('{END_DATE}'), interval 1 day)) AS date")
    .withColumn("date_key", F.date_format(F.col("date"), "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("month_short", F.date_format("date", "MMM"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("year_quarter", F.concat_ws("-Q", F.col("year"), F.col("quarter")))
    .withColumn("day_of_month", F.dayofmonth("date"))
    # ISO weekday (Mon=1..Sun=7) without date_format('u')
    .withColumn("day_of_week_iso", ((F.dayofweek("date") + F.lit(5)) % F.lit(7) + F.lit(1)).cast("int"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("day_name_short", F.date_format("date", "EEE"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("is_weekend", F.col("day_of_week_iso").isin([6,7]).cast("boolean"))
    .withColumn("is_month_start", (F.col("date") == F.trunc("date", "month")).cast("boolean"))
    .withColumn("is_month_end", (F.col("date") == F.last_day("date")).cast("boolean"))
)

years = df_date.select("year").distinct()
y = F.col("year")

def month_start(col_year, month_num):
    return F.to_date(F.concat_ws("-", col_year.cast("string"), F.lit(f"{month_num:02d}"), F.lit("01")))

def observed_date(d):
    # dayofweek: 1=Sun..7=Sat
    return (F.when(F.dayofweek(d) == 7, F.date_sub(d, 1))   # Sat -> Fri
             .when(F.dayofweek(d) == 1, F.date_add(d, 1))   # Sun -> Mon
             .otherwise(d))

def nth_weekday_of_month(first_day, weekday_name, n):
    first = F.next_day(F.date_sub(first_day, 1), weekday_name)
    return F.date_add(first, 7 * (n - 1))

def last_weekday_of_month(any_day_in_month, weekday_name):
    ld = F.last_day(any_day_in_month)
    return F.date_sub(F.next_day(ld, weekday_name), 7)

# Holiday table (US federal holidays, observed)
h = years.select(
    y.alias("year"),
    observed_date(month_start(y, 1)).alias("new_years"),
    nth_weekday_of_month(month_start(y, 1), "Mon", 3).alias("mlk_day"),
    nth_weekday_of_month(month_start(y, 2), "Mon", 3).alias("presidents_day"),
    last_weekday_of_month(month_start(y, 5), "Mon").alias("memorial_day"),
    observed_date(F.to_date(F.concat_ws("-", y.cast("string"), F.lit("06"), F.lit("19")))).alias("juneteenth"),
    observed_date(F.to_date(F.concat_ws("-", y.cast("string"), F.lit("07"), F.lit("04")))).alias("independence_day"),
    nth_weekday_of_month(month_start(y, 9), "Mon", 1).alias("labor_day"),
    nth_weekday_of_month(month_start(y, 10), "Mon", 2).alias("columbus_day"),
    observed_date(F.to_date(F.concat_ws("-", y.cast("string"), F.lit("11"), F.lit("11")))).alias("veterans_day"),
    nth_weekday_of_month(month_start(y, 11), "Thu", 4).alias("thanksgiving"),
    observed_date(F.to_date(F.concat_ws("-", y.cast("string"), F.lit("12"), F.lit("25")))).alias("christmas")
)

df_holidays = (
    h.selectExpr(
        "stack(11, "
        "new_years, 'New Year''s Day', "
        "mlk_day, 'Martin Luther King Jr. Day', "
        "presidents_day, 'Washington''s Birthday', "
        "memorial_day, 'Memorial Day', "
        "juneteenth, 'Juneteenth National Independence Day', "
        "independence_day, 'Independence Day', "
        "labor_day, 'Labor Day', "
        "columbus_day, 'Columbus Day', "
        "veterans_day, 'Veterans Day', "
        "thanksgiving, 'Thanksgiving Day', "
        "christmas, 'Christmas Day'"
        ") as (date, us_federal_holiday_name)"
    )
    .withColumn("date", F.col("date").cast("date"))
    # Juneteenth is federal from 2021 onwards
    .filter(~((F.col("us_federal_holiday_name") == "Juneteenth National Independence Day") & (F.year("date") < 2021)))
)

df_date2 = (
    df_date.join(df_holidays, on="date", how="left")
           .withColumn("is_us_federal_holiday", F.col("us_federal_holiday_name").isNotNull())
)

(
    df_date2.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dim_date")
)

#display(df_date2.filter("is_us_federal_holiday").orderBy("date"))

## Time of day dimension

Creates a small Time of Day dimension (1,440 rows, one per minute).

This dimension enables analysis by:
- Hour of day
- AM / PM
- Rush hour vs off-peak
- Night vs daytime travel

It can be joined to the fact table using a derived time key
from pickup or dropoff datetime.

In [ ]:
# Generate one row per minute of day (00:00 – 23:59)
df_time = (
    spark.range(0, 24 * 60)
         .withColumn("hour", (F.col("id") / 60).cast("int"))
         .withColumn("minute", (F.col("id") % 60).cast("int"))
         .drop("id")
)

df_time = (
    df_time
    .withColumn("time_key", (F.col("hour") * 100 + F.col("minute")).cast("int"))
    .withColumn("hour_label", F.format_string("%02d", F.col("hour")))
    .withColumn("minute_label", F.format_string("%02d", F.col("minute")))
    .withColumn("time_label", F.concat_ws(":", "hour_label", "minute_label"))
    .withColumn("am_pm", F.when(F.col("hour") < 12, "AM").otherwise("PM"))

    # Part of day buckets (tunable)
    .withColumn(
        "part_of_day",
        F.when(F.col("hour") < 6, "Night")
         .when(F.col("hour") < 12, "Morning")
         .when(F.col("hour") < 18, "Afternoon")
         .otherwise("Evening")
    )

    # Rush hour definition (NYC typical)
    .withColumn(
        "is_rush_hour",
        (
            (F.col("hour").between(7, 9)) |
            (F.col("hour").between(16, 18))
        ).cast("boolean")
    )

    .withColumn(
        "is_night",
        (F.col("hour") < 6).cast("boolean")
    )
)

(
    df_time.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dim_time_of_day")
)

#display(df_time.orderBy("time_key"))

## Validation

Basic checks to ensure dimension keys fully cover the values present in the fact table.

In [ ]:
# ----------------------------
# Optional validation: show distinct keys present in fact vs dimensions
# ----------------------------
fact = spark.table(FACT_TABLE)

print("Distinct VendorID in fact:", fact.select("VendorID").distinct().count())
print("Distinct RatecodeID in fact:", fact.select("RatecodeID").distinct().count())
print("Distinct payment_type in fact:", fact.select("payment_type").distinct().count())

zones = spark.table("dim_taxi_zone")
missing_pu = (
    fact.select(F.col("PULocationID").alias("LocationID"))
        .where(F.col("LocationID").isNotNull())
        .distinct()
        .join(zones, on="LocationID", how="left_anti")
        .count()
)
missing_do = (
    fact.select(F.col("DOLocationID").alias("LocationID"))
        .where(F.col("LocationID").isNotNull())
        .distinct()
        .join(zones, on="LocationID", how="left_anti")
        .count()
)

print("Missing pickup zones (not in dim_taxi_zone):", missing_pu)
print("Missing dropoff zones (not in dim_taxi_zone):", missing_do)